# ALS-Based Movie Recommendation Engine

### - Project Objective

The goal of this project is to build a personalized movie recommendation system using collaborative filtering techniques on Databricks Free Edition.

The system predicts:

“Which movies is a user most likely to rate highly?”

This is achieved using the ALS (Alternating Least Squares) algorithm from Apache Spark MLlib.

## Full Pipeline Architecture

Bronze Layer → Raw CSV data

Silver Layer → Cleaned & structured data

ML Layer → ALS model training

Evaluation Layer → RMSE performance validation

Recommendation Layer → Manual generation of top-N movies

Presentation Layer → Join with movie titles




#🥉 Bronze Layer (Data Ingestion)
Load MovieLens datasets:
Ratings (userId, movieId, rating)
Movies (movieId, title, genres)
Store raw data as-is
No cleaning or transformation
Schema inferred automatically














In [0]:
# Load ratings data
ratings_df = spark.read \
    .option("header", True) \
    .option("inferSchema", True) \
    .csv("/Volumes/workspace/default/movie_volume/ratings.csv")

# Load movies data
movies_df = spark.read \
    .option("header", True) \
    .option("inferSchema", True) \
    .csv("/Volumes/workspace/default/movie_volume/movies.csv")

ratings_df.show(5)
movies_df.show(5)


#🥈 Silver Layer (Data Cleaning)
Select required columns: userId, movieId, rating
Convert data types:
userId, movieId → int
rating → float
Remove null values

✔ Output: Clean, structured data ready for ML

In [0]:
from pyspark.sql.functions import col

ratings_clean = ratings_df \
    .select(
        col("userId").cast("int"),
        col("movieId").cast("int"),
        col("rating").cast("float")
    ) \
    .dropna()

movies_clean = movies_df \
    .select(
        col("movieId").cast("int"),
        col("title"),
        col("genres")
    ) \
    .dropna()

ratings_clean.printSchema()
movies_clean.printSchema()

#🤖 Model Building (ALS)
Use ALS (collaborative filtering)

Learns:
User preferences
Movie features

Key configs:
userCol, itemCol, ratingCol
coldStartStrategy = "drop"
nonnegative = True

🧪 Train-Test Split
80% training
20% testing

✔ Ensures proper evaluation on unseen data

In [0]:
from pyspark.ml.recommendation import ALS
from pyspark.ml.evaluation import RegressionEvaluator

# Split data
train, test = ratings_clean.randomSplit([0.8, 0.2])

# Define ALS model
als = ALS(
    userCol="userId",
    itemCol="movieId",
    ratingCol="rating",
    coldStartStrategy="drop",
    nonnegative=True
)

# Train
model = als.fit(train)

# Predict
predictions = model.transform(test)


#📊 Model Evaluation
Metric: RMSE

Lower RMSE = better accuracy

In [0]:
evaluator = RegressionEvaluator(
    metricName="rmse",
    labelCol="rating",
    predictionCol="prediction"
)

rmse = evaluator.evaluate(predictions)
print("RMSE:", rmse)


#🎯 Recommendation Generation (Manual)
(Used due to Free Edition limitations)

- Get all users
- Get all movies
- Create all user–movie pairs
- Predict ratings using ALS
- Rank movies per user
- Select Top 5 recommendations per user

In [0]:
users = ratings_clean.select("userId").distinct()
movies_df = ratings_clean.select("movieId").distinct()

from pyspark.sql.functions import col

user_movie = users.crossJoin(movies_df)



In [0]:
predictions = model.transform(user_movie)


In [0]:
from pyspark.sql.window import Window
from pyspark.sql.functions import row_number

window = Window.partitionBy("userId").orderBy(col("prediction").desc())

top_recommendations = predictions.withColumn(
    "rank",
    row_number().over(window)
).filter(col("rank") <= 5)

top_recommendations.show()


In [0]:
final = top_recommendations.join(movies_clean, "movieId")

final.select("userId", "title", "prediction").show(truncate=False)


#🚀 Final Result

A complete pipeline that:

- Ingests raw data
- Cleans & prepares it
- Trains ML model
- Evaluates accuracy
- Generates personalized recommendations